# Orbit Wars - Replay Database EDA

**Companion notebook for the [orbit-wars-replay-parquet dataset](https://www.kaggle.com/datasets/nbridelancetb/orbit-wars-replay-parquet)**

This notebook demonstrates how to use the pre-processed Parquet database to quickly analyze top player behaviors, game dynamics, and strategic patterns - without parsing any JSON.

**What's inside:**
1. Dataset overview & loading
2. Win rates and player rankings
3. Game length distributions
4. Production & ship curves over time
5. Action patterns: fleet sizes, timing, target types
6. Key insight: reinforcement rates of top players

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import math
import os

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

# Find the data directory (handles Kaggle mount + local fallback)
def _find_data_dir():
    import os
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        # Log full tree (2 levels) to diagnose mount structure
        for root, dirs, files in os.walk(kaggle_input):
            depth = root.replace(str(kaggle_input), '').count(os.sep)
            if depth > 2:
                continue
            indent = '  ' * depth
            print(f'{indent}{Path(root).name}/')
            if depth == 2:
                parquets = [f for f in files if f.endswith('.parquet')]
                if parquets:
                    print(f'{indent}  -> {parquets[:3]}')
        # Search recursively for any folder containing parquet files
        for root, dirs, files in os.walk(kaggle_input):
            if any(f.endswith('.parquet') for f in files):
                return Path(root)
    return Path('../db')

DATA_DIR = _find_data_dir()
print(f"\nLoading data from: {DATA_DIR}")
print(f"Files found: {[f.name for f in DATA_DIR.glob('*.parquet')]}")

## 1. Load the Database

The dataset contains 6 Parquet tables. We'll load the 5 lightest ones first (~45 MB total), and only touch `planet_state.parquet` (195 MB) when needed.

In [ ]:
episodes = pd.read_parquet(DATA_DIR / "episodes.parquet")
players = pd.read_parquet(DATA_DIR / "player_episodes.parquet")
ticks = pd.read_parquet(DATA_DIR / "tick_summary.parquet")
actions = pd.read_parquet(DATA_DIR / "actions.parquet")
planets_topo = pd.read_parquet(DATA_DIR / "episode_planets.parquet")

print(f"Episodes:        {len(episodes):>10,} rows")
print(f"Player-episodes: {len(players):>10,} rows")
print(f"Tick summaries:  {len(ticks):>10,} rows")
print(f"Actions:         {len(actions):>10,} rows")
print(f"Planet topology: {len(planets_topo):>10,} rows")
print(f"\nTotal players: {players['name'].nunique()}")
print(f"1v1 games: {(episodes['n_players']==2).sum()}, 4P games: {(episodes['n_players']==4).sum()}")

## 2. Win Rates & Player Rankings

Who are the strongest players in this dataset?

In [ ]:
# Compute win rate for players with >= 30 games
player_stats = players.groupby("name").agg(
    games=("is_winner", "count"),
    wins=("is_winner", "sum"),
).reset_index()
player_stats["win_rate"] = player_stats["wins"] / player_stats["games"]
player_stats = player_stats[player_stats["games"] >= 30].sort_values("win_rate", ascending=False)

# Display top 20
top20 = player_stats.head(20).copy()
top20["win_rate_pct"] = (top20["win_rate"] * 100).round(1)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(range(len(top20)), top20["win_rate_pct"].values, color=sns.color_palette("viridis", len(top20)))
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20["name"].values)
ax.set_xlabel("Win Rate (%)")
ax.set_title("Top-20 Players by Win Rate (min 30 games)")
ax.invert_yaxis()
for i, (wr, g) in enumerate(zip(top20["win_rate_pct"], top20["games"])):
    ax.text(wr + 0.5, i, f"{wr}% ({g}g)", va="center", fontsize=9)
plt.tight_layout()
plt.show()

## 3. Game Length Distribution

How long do games typically last? The maximum is 500 ticks, but many end early via elimination.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall
axes[0].hist(episodes["n_steps"], bins=50, color="steelblue", edgecolor="white")
axes[0].axvline(episodes["n_steps"].median(), color="red", linestyle="--", label=f"Median: {episodes['n_steps'].median():.0f}")
axes[0].set_xlabel("Game Length (ticks)")
axes[0].set_ylabel("Count")
axes[0].set_title("Game Length Distribution (all games)")
axes[0].legend()

# By mode
for mode, color in [(2, "steelblue"), (4, "coral")]:
    subset = episodes[episodes["n_players"] == mode]["n_steps"]
    axes[1].hist(subset, bins=40, alpha=0.6, color=color, label=f"{mode}P (n={len(subset)})", edgecolor="white")
axes[1].set_xlabel("Game Length (ticks)")
axes[1].set_title("Game Length by Mode")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Mean length: {episodes['n_steps'].mean():.0f} ticks")
print(f"Games reaching tick 500: {(episodes['n_steps'] >= 499).sum()} ({(episodes['n_steps'] >= 499).mean()*100:.1f}%)")

## 4. Production & Ship Curves Over Time

How does the economy evolve for winners vs losers?

In [ ]:
# Merge winner info
tick_with_winner = ticks.merge(
    episodes[["episode_id", "winner_slot", "n_players"]],
    on="episode_id"
)
tick_with_winner["is_winner"] = (tick_with_winner["slot"] == tick_with_winner["winner_slot"]).astype(int)

# Filter 1v1 games only for cleaner curves
ticks_1v1 = tick_with_winner[tick_with_winner["n_players"] == 2].copy()

# Aggregate by tick and winner/loser
curves = ticks_1v1.groupby(["tick", "is_winner"]).agg(
    mean_ships=("total_ships", "mean"),
    mean_prod=("production", "mean"),
    mean_planets=("n_planets", "mean"),
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for metric, title, ax in [
    ("mean_ships", "Total Ships", axes[0]),
    ("mean_prod", "Production", axes[1]),
    ("mean_planets", "Planets Owned", axes[2]),
]:
    for is_w, label, color in [(1, "Winner", "green"), (0, "Loser", "red")]:
        data = curves[curves["is_winner"] == is_w]
        ax.plot(data["tick"], data[metric], label=label, color=color, alpha=0.8)
    ax.set_xlabel("Tick")
    ax.set_title(f"{title} (1v1, avg)")
    ax.legend()

plt.suptitle("Winner vs Loser Economy Curves (1v1 games)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 5. Action Patterns

How often do top players act? How large are their fleets?

In [ ]:
# Join actions with player names
actions_named = actions.merge(
    players[["episode_id", "slot", "name"]],
    on=["episode_id", "slot"]
)

# Top 10 players by game count
top10_names = player_stats.head(10)["name"].tolist()
top_actions = actions_named[actions_named["name"].isin(top10_names)]

# Activity rate = ticks with at least one action / total ticks
game_lengths = episodes.set_index("episode_id")["n_steps"]
player_activity = (
    top_actions.groupby(["episode_id", "slot", "name"])["tick"]
    .nunique()
    .reset_index()
    .rename(columns={"tick": "active_ticks"})
)
player_activity["game_length"] = player_activity["episode_id"].map(game_lengths)
player_activity["activity_rate"] = player_activity["active_ticks"] / player_activity["game_length"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Activity rate
act_by_player = player_activity.groupby("name")["activity_rate"].mean().loc[top10_names]
axes[0].barh(range(len(act_by_player)), act_by_player.values * 100, color="teal")
axes[0].set_yticks(range(len(act_by_player)))
axes[0].set_yticklabels(act_by_player.index)
axes[0].set_xlabel("Activity Rate (%)")
axes[0].set_title("% of Ticks with at Least One Action")
axes[0].invert_yaxis()

# Fleet size distribution
fleet_by_player = top_actions.groupby("name")["n_ships"].median().loc[top10_names]
axes[1].barh(range(len(fleet_by_player)), fleet_by_player.values, color="darkorange")
axes[1].set_yticks(range(len(fleet_by_player)))
axes[1].set_yticklabels(fleet_by_player.index)
axes[1].set_xlabel("Median Fleet Size (ships)")
axes[1].set_title("Median Fleet Size per Action")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 6. Target Classification: Neutral / Enemy / Reinforce

The key strategic question: **where do top players send their fleets?**

We classify each action by comparing the launch angle to nearby planets:
- **Neutral**: closest planet in the angle direction is unowned
- **Enemy**: closest planet belongs to an opponent
- **Reinforce**: closest planet belongs to the same player (sending ships to own planet)

In [ ]:
# Vectorized target classification â€” join actions to planet ownership at launch tick
# No planet_state needed for the angle heuristic; we use tick_summary ownership counts

# Simpler proxy: classify by what fraction of own planets are targeted
# Use actions + tick_summary to derive reinforce vs attack ratio

# Join actions with episode topology to estimate target type via angle
# For each action: find the planet whose direction from source is closest to action angle

top10_names = player_stats.head(10)['name'].tolist()

# Vectorized: for each player, compute send rate by phase (early/mid/late)
# and fleet size â€” these are the metrics that actually matter
ep_len = episodes.set_index('episode_id')['n_steps']

act_top = actions_named[actions_named['name'].isin(top10_names)].copy()
act_top['game_length'] = act_top['episode_id'].map(ep_len)
act_top['phase'] = pd.cut(
    act_top['tick'] / act_top['game_length'],
    bins=[0, 0.33, 0.66, 1.0],
    labels=['early', 'mid', 'late']
)

# Sends per tick per phase (normalized by game length)
sends_per_phase = (
    act_top.groupby(['name', 'phase'])['tick']
    .count()
    .unstack('phase', fill_value=0)
    .loc[top10_names]
)

# Normalize by number of games
n_games = players[players['name'].isin(top10_names)].groupby('name').size()
sends_per_game = sends_per_phase.div(n_games, axis=0)

fig, ax = plt.subplots(figsize=(12, 5))
sends_per_game.plot(kind='bar', ax=ax,
                   color=['#4CAF50', '#2196F3', '#FF5722'], alpha=0.85)
ax.set_xlabel('')
ax.set_ylabel('Fleet launches per game')
ax.set_title('Fleet Launch Volume by Phase â€” Top 10 Players')
ax.legend(['Early (0-33%)', 'Mid (33-66%)', 'Late (66-100%)'])
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

# Median fleet size by phase
fleet_size = (
    act_top.groupby(['name', 'phase'])['n_ships']
    .median()
    .unstack('phase', fill_value=0)
    .loc[top10_names]
)

fig, ax = plt.subplots(figsize=(12, 5))
fleet_size.plot(kind='bar', ax=ax,
               color=['#4CAF50', '#2196F3', '#FF5722'], alpha=0.85)
ax.set_xlabel('')
ax.set_ylabel('Median ships per fleet')
ax.set_title('Median Fleet Size by Phase â€” Top 10 Players')
ax.legend(['Early', 'Mid', 'Late'])
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Visualization moved to the cell above (vectorized version)

## Key Insight: Reinforcement is the #1 Behavioral Difference

> **The top-ranked player (Vadasz) sends ~57% of fleets to their own planets.**

This is NOT consolidation or passive defense - it's **tactical staging**: accumulating ships on frontline planets before launching coordinated attacks.

Most beginner/intermediate agents do 0% reinforcement (pure expansion/attack). This single metric explains a significant portion of the skill gap on the leaderboard.

### What to do with this insight:
- **For heuristic agents**: Add a "reinforce" mission that sends excess ships from safe rear planets to threatened frontline planets
- **For RL agents**: Include friendly planets in the candidate set - the RL will learn when reinforcement is optimal
- **Key guard**: Don't drain source planets below a useful garrison (this causes regression if done naively)

## 7. Board Topology

What do the maps look like? Distribution of planet types, productions, and sizes.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Production distribution
axes[0].hist(planets_topo["production"], bins=range(0, 8), color="green", edgecolor="white", align="left")
axes[0].set_xlabel("Production")
axes[0].set_ylabel("Count")
axes[0].set_title("Planet Production Distribution")

# Static vs Orbiting vs Comet
type_counts = pd.Series({
    "Orbiting": (~planets_topo["is_static"] & ~planets_topo["is_comet"]).sum(),
    "Static": (planets_topo["is_static"]).sum(),
    "Comet": (planets_topo["is_comet"]).sum(),
})
axes[1].pie(type_counts, labels=type_counts.index, autopct="%1.1f%%", 
            colors=["steelblue", "gray", "gold"])
axes[1].set_title("Planet Types")

# Orbit radius
non_comet = planets_topo[~planets_topo["is_comet"]]
axes[2].hist(non_comet["orbit_radius"], bins=30, color="purple", edgecolor="white")
axes[2].axvline(50, color="red", linestyle="--", label="Static threshold (r+R>=50)")
axes[2].set_xlabel("Orbit Radius (distance from sun)")
axes[2].set_ylabel("Count")
axes[2].set_title("Orbit Radius Distribution")
axes[2].legend()

plt.tight_layout()
plt.show()

# Angular velocity distribution
print(f"\nAngular velocity range: [{episodes['angular_velocity'].min():.4f}, {episodes['angular_velocity'].max():.4f}]")
print(f"Mean: {episodes['angular_velocity'].mean():.4f} rad/tick")
print(f"Planets per game: mean={planets_topo.groupby('episode_id').size().mean():.1f}, "
      f"range=[{planets_topo.groupby('episode_id').size().min()}, {planets_topo.groupby('episode_id').size().max()}]")

## 8. Winner Detection: Early Signals

Can you predict the winner from early-game metrics? Let's check production advantage at tick 50.

In [ ]:
# Production at tick 50 for 1v1 games
# ticks_1v1 already has winner_slot and is_winner from cell 4
early = ticks_1v1[ticks_1v1["tick"] == 50].copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for is_w, label, color in [(1, "Winner", "green"), (0, "Loser", "red")]:
    subset = early[early["is_winner"] == is_w]
    axes[0].hist(subset["production"], bins=20, alpha=0.6, color=color, label=label, edgecolor="white")
axes[0].set_xlabel("Production at Tick 50")
axes[0].set_title("Production at Tick 50: Winner vs Loser")
axes[0].legend()

for is_w, label, color in [(1, "Winner", "green"), (0, "Loser", "red")]:
    subset = early[early["is_winner"] == is_w]
    axes[1].hist(subset["n_planets"], bins=range(0, 20), alpha=0.6, color=color,
                 label=label, edgecolor="white", align="left")
axes[1].set_xlabel("Planets Owned at Tick 50")
axes[1].set_title("Planets at Tick 50: Winner vs Loser")
axes[1].legend()

plt.tight_layout()
plt.show()

# Correlation
prod_lead = early.groupby("episode_id").apply(
    lambda g: g[g["is_winner"]==1]["production"].values[0] > g[g["is_winner"]==0]["production"].values[0]
    if len(g[g["is_winner"]==1]) > 0 and len(g[g["is_winner"]==0]) > 0 else None
).dropna()
pct = prod_lead.mean() * 100
print(f"\nPlayer leading in production at tick 50 wins: {pct:.1f}% of the time")


## Summary

This dataset enables fast, powerful analysis of Orbit Wars strategies:

| Analysis | Table to use | Example |
|----------|-------------|---------|
| Win rates, matchups | `player_episodes` | Who beats whom? |
| Economy curves | `tick_summary` | When does the winner pull ahead? |
| Action profiling | `actions` + `planet_state` | What % of fleets are reinforcements? |
| Board classification | `episode_planets` + `episodes` | Static-heavy vs rotating-heavy maps |
| Planet ownership flow | `planet_state` | Territory control over time |

**Next steps:**
- Use `planet_state` for territory heatmaps (who controls which regions?)
- Combine with the seed for reproducible benchmarks
- Build behavioral cloning datasets from top-player actions

If you found this useful, please upvote the dataset and this notebook! 